# Run your dataset

The [quickstart](quickstart.ipynb) taught you to read the checkpoints and **choose
the two embedding parameters** `(τ, m)` on a single file. This notebook takes the
next step: point the **standard pipeline** at a *folder* of canonical CSVs and get a
tidy table of metrics out — both movement **magnitude** (linear) and **organization**
(recurrence), for every keypoint.

You edit a **config**, not code. It runs on a bundled example folder first; then you
change one line to your own data.

## The config — edit this

Everything the run needs is here. The important values are your **frame rate** and
the `(τ, m)` you committed in the quickstart. The rest have sensible defaults (see
the [configuration reference](../docs/configuration.md)).

In [ ]:
from pose_dynamics import StudyConfig, run_study
from pose_dynamics.data import example_dataset

config = StudyConfig(
    data       = str(example_dataset(3)),   # <-- a bundled 3-trial example folder
    # data     = "/work/data",              # <-- SWAP: your folder of canonical CSVs
    frame_rate = 60.0,                       # <-- Hz, how your data was recorded

    tau        = 24,                         # <-- from the quickstart (embedding delay)
    m          = 3,                          # <-- from the quickstart (embedding dimension)

    # Feature pipeline (optional). Uncomment to extract ROI / kinematic / aperture
    # features instead of per-keypoint speed — see docs/feature_steps.md.
    # features = [
    #     {"primitive": "coordinate_normalization", "params": {"width": 720, "height": 720}},
    #     {"primitive": "roi_centroid", "params": {"rois": {"arms": [2,3,4,5,6,7]}}},
    #     {"primitive": "velocity_magnitude", "params": {"method": "diff"}},
    # ],
    # features_dir = "features",              # write the feature time series per file
    window_s   = 30.0,                        # analysis window (seconds); None = whole trial
    radius_mode = "fixed_rrec",               # target a recurrence rate; the radius is solved
    target_rec = 5.0,                         # % (see the note on choosing this below)
)
config.to_dict()

## Run it

For each trial the pipeline masks low-confidence points, interpolates short gaps,
filters, checks data quality, then — per keypoint, per window — computes the linear
summaries and the recurrence metrics.

In [ ]:
results, quality = run_study(config, progress=True)
print(f"{len(results)} rows  ({results['file'].nunique()} trials x keypoints x windows)")

## First: the data-quality report

**Read this before the metrics.** One row per trial. `status` is `ok`, `flag`, or
`exclude`; `missing_fraction` is how much data was still missing after
interpolation. Nothing is dropped silently — a flagged or excluded trial appears
here so you can decide what to do. If trials are being excluded, revisit your
confidence threshold or whether those recordings are usable at all.

In [ ]:
quality[["source_file", "status", "missing_fraction"]]

## The metrics table

One row per **file × keypoint × window**, with two families of columns:

- **Magnitude (linear):** `speed_mean`, `speed_std`, `speed_rms`, `speed_max` — how
  much and how fast the keypoint moved.
- **Organization (recurrence):** `perc_recur` (RR), `perc_determ` (DET),
  `laminarity`, `mean_line_length`, `maxl_found` (Lmax), `entropy` — how that
  movement was structured in time. `radius_used` is the achieved threshold.

This is the tidy output you take into your own statistics (the package computes no
inferential statistics itself).

In [ ]:
results.head(10)

In [ ]:
# save it
results.to_csv("study_results.csv", index=False)
print("wrote study_results.csv")

## Running without a notebook (a config file + one command)

For a real project you'll want to run this over a folder repeatedly and keep the
settings under version control. Save the config to a file and run it from the
terminal — no notebook, no Python:

```bash
pose-dynamics new-config study.yaml     # writes a template to edit
pose-dynamics run study.yaml            # extracts features + metrics over the folder
```

Ready-to-edit examples are in [`configs/`](../configs/). A config declares the whole
run — including the **feature pipeline** — so you can extract ROI, aperture, or
kinematic features across a dataset by editing text, not code. You can also load one
here with `config = StudyConfig.from_file("study.yaml")`.

## Which metrics should you look at?

You now have many metrics — don't test all of them blindly. The paper's guidance:

- A practical **core set** is **RR, DET, and Lmax** (or mean line length): recurrence
  density, predictability, and the length of sustained structured episodes.
- Add **laminarity / trapping time** when *persistence or intermittency* is your
  question, and **entropy** when the *diversity of timescales* is.
- **Interpret them together**, not in isolation — they are mathematically coupled.
  High RR with low DET means noise-like recurrence; low RR with high DET means rare
  but highly structured returns.
- **Decide in advance** which metrics you expect your manipulation to move, and on
  what grounds; report anything beyond that as exploratory.
- **Absolute values aren't meaningful on their own** — they shift with `(τ, m)` and
  the radius. What should be stable is the *comparative* pattern across your
  conditions. If you used `fixed_rrec` (a target %REC), then %REC is pinned by design
  and the **radius** carries the density signal instead.

## Use your own data

In the config cell, comment out the `example_dataset(...)` line and set
`data = "/work/data"` (your CSVs in the mounted folder, in Docker) or any folder
path. Make sure `frame_rate` matches your recording, and keep the `(τ, m)` you
committed in the quickstart. Re-run.